In [19]:
import pickle
import pandas as pd

In [107]:
tbl_co = '/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/find_regions_recomb/tbl_co.txt'
regions='/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/find_regions_recomb/regions.txt'
tract='/data/proj2/home/students/u.srinivasan/Plasmodium_us/Inputs/SMC/find_regions_recomb/conversion_tract.txt'

In [108]:
tbl_co= pd.read_csv(tbl_co, sep='\t')

regions= pd.read_csv(regions, sep='\t', names=['region_chrom', 'region_start', 'region_stop', 'region_type'])

In [ ]:

regions.head()


,region_chrom,region_start,region_stop,region_type
0,Pf3D7_01_v3,1,27336,SubtelomericRepeat
1,Pf3D7_01_v3,27337,92900,SubtelomericHypervariable
2,Pf3D7_01_v3,92901,457931,Core
3,Pf3D7_01_v3,457932,460311,Centromere
4,Pf3D7_01_v3,460312,575900,Core


In [ ]:
regions = regions[regions['region_type'] == 'Core']


In [50]:
tbl_marker_span = (
    regions
    .groupby('region_chrom')  # Group by 'region_chrom'
    .agg(start=('region_start', 'min'),  # Aggregation for 'start'
         stop=('region_stop', 'max'))   # Aggregation for 'stop'
    .reset_index()  # Ensure 'region_chrom' becomes a column again (optional)
    .rename(columns={'region_chrom': 'chrom'})  # Rename column for clarity
    .assign(span=lambda df: df.stop - df.start)  # Add 'span' column
)
tbl_marker_span.head()

,chrom,start,stop,span
0,Pf3D7_01_v3,92901,575900,482999
1,Pf3D7_02_v3,105801,862500,756699
2,Pf3D7_03_v3,70631,1003060,932429
3,Pf3D7_04_v3,91421,1143990,1052569
4,Pf3D7_05_v3,37901,1321390,1283489


In [96]:
tbl_co.head()

,sample,chrom,co_pos_mid,co_pos_min,co_pos_max,co_pos_range,cross,co_from_parent,co_to_parent
0,B1SD/PG0015-C/ERR019044,b'Pf3D7_01_v3',145052,144877,145227,350,hb3_dd2,hb3,dd2
1,GC03/PG0021-C/ERR015447,b'Pf3D7_01_v3',163584,163145,164024,879,hb3_dd2,dd2,hb3
2,XF12/PG0102-C/ERR029143,b'Pf3D7_01_v3',206769,205803,207736,1933,7g8_gb4,gb4,7g8
3,7C159/PG0040-Cx/ERR107475,b'Pf3D7_01_v3',206905,206074,207736,1662,hb3_dd2,hb3,dd2
4,CH3_61/PG0033-Cx/ERR175544,b'Pf3D7_01_v3',206905,206074,207736,1662,hb3_dd2,dd2,hb3


In [112]:
# Set option to display all rows/columns
pd.set_option('display.max_rows', None)  # Set to None to display all rows
pd.set_option('display.max_columns', None)  # Set to None to display all columns



In [121]:
#tbl_co.groupby(['chrom', 'sample','cross'])['chrom'].count()
counts=tbl_co.groupby(['chrom', 'cross','sample'])['chrom'].count()

# Step 2: Group by (cross, sample) and calculate the average number of rows per group
avg_rows_per_cross_sample = counts.groupby(['chrom','cross']).mean()

# Step 3: Print the result
print(avg_rows_per_cross_sample.groupby(['chrom']).mean())


chrom
b'Pf3D7_01_v3'    1.303175
b'Pf3D7_02_v3'    1.544444
b'Pf3D7_03_v3'    1.702614
b'Pf3D7_04_v3'    1.461765
b'Pf3D7_05_v3'    1.408871
b'Pf3D7_06_v3'    1.661111
b'Pf3D7_07_v3'    1.471958
b'Pf3D7_08_v3'    1.493961
b'Pf3D7_09_v3'    1.638519
b'Pf3D7_10_v3'    1.686232
b'Pf3D7_11_v3'    2.053080
b'Pf3D7_12_v3'    1.815764
b'Pf3D7_13_v3'    2.351603
b'Pf3D7_14_v3'    2.599746
Name: chrom, dtype: float64


In [76]:
# Normalize counts by dividing by the number of unique 'sample' values for each (chrom, cross) group
normalized_counts = tbl_co.groupby(['chrom', 'cross']).size() / tbl_co.groupby(['chrom', 'cross'])['sample'].nunique()

# Now, group by 'chrom' to find the mean of these normalized counts
mean_per_chrom = normalized_counts.groupby('chrom').mean().reset_index()
mean_per_chrom.rename(columns={0: 'No_co'}, inplace=True)
print(mean_per_chrom)

             chrom     No_co
0   b'Pf3D7_01_v3'  1.303175
1   b'Pf3D7_02_v3'  1.544444
2   b'Pf3D7_03_v3'  1.702614
3   b'Pf3D7_04_v3'  1.461765
4   b'Pf3D7_05_v3'  1.408871
5   b'Pf3D7_06_v3'  1.661111
6   b'Pf3D7_07_v3'  1.471958
7   b'Pf3D7_08_v3'  1.493961
8   b'Pf3D7_09_v3'  1.638519
9   b'Pf3D7_10_v3'  1.686232
10  b'Pf3D7_11_v3'  2.053080
11  b'Pf3D7_12_v3'  1.815764
12  b'Pf3D7_13_v3'  2.351603
13  b'Pf3D7_14_v3'  2.599746


In [77]:
mean_per_chrom['span'] = tbl_marker_span['span']

mean_per_chrom['co_cm'] = (mean_per_chrom['No_co'] / (mean_per_chrom['span'] / 1e6))


In [79]:
mean_per_chrom

,chrom,No_co,span,co_cm
0,b'Pf3D7_01_v3',1.303175,482999,2.698090
1,b'Pf3D7_02_v3',1.544444,756699,2.041029
2,b'Pf3D7_03_v3',1.702614,932429,1.825999
3,b'Pf3D7_04_v3',1.461765,1052569,1.388759
4,b'Pf3D7_05_v3',1.408871,1283489,1.097689
5,b'Pf3D7_06_v3',1.661111,1222479,1.358805
6,b'Pf3D7_07_v3',1.471958,1304499,1.128370
7,b'Pf3D7_08_v3',1.493961,1292169,1.156166
8,b'Pf3D7_09_v3',1.638519,1394459,1.175021
9,b'Pf3D7_10_v3',1.686232,1502844,1.122027


In [ ]:
(tbl_co
 .groupby('chrom') 
  .addfield('count_per_meiosis', 
            lambda row: row['count'] / n_progeny[row['cross']])
 .display(caption='CO events by cross')
)

AttributeError: 'DataFrame' object has no attribute 'valuecounts'

In [93]:
full_tracts=  pd.read_csv(tract, sep='\t')
#full_tracts[full_tracts['tract_type']=='CO'].head()

In [95]:
full_tracts.groupby(['chrom', 'cross','tract_type','sample']).size()

chrom           cross    tract_type  sample                  
b'Pf3D7_01_v3'  3d7_hb3  CO          C12/PG0058-C/ERR019063      1
                7g8_gb4  NCO         DAN/PG0098-C/ERR027110      1
                                     DEV/PG0081-CW/ERR045633     1
                                     NF10/PG0096-C/ERR027108     2
                hb3_dd2  NCO         7C183/PG0042-C/ERR015448    1
                                                                ..
b'Pf3D7_14_v3'  hb3_dd2  NCO         GC03/PG0021-C/ERR015447     1
                                     GC06/PG0028-C/ERR015456     1
                                     QC01/PG0017-C/ERR019050     1
                                     TC05/PG0027-C/ERR015450     1
                                     TC08/PG0020-C/ERR019052     1
Length: 297, dtype: int64